In [12]:
from pprint import pprint
from collections import Counter, defaultdict
import re
import json

from matplotlib import pyplot
import pandas as pd

from brickmapper import Mapper
from brickmapper.bedes_parser.bedes_parser import BedesParser
from brickmapper.buildingsync.buildingsync_parser import BuildingSyncParser


In [13]:
"""Get BEDES defns"""
bedes = BedesParser('v2.5')
bedes_defns = bedes._get_term_definitions() + bedes._get_enumeration_definitions()

pprint(bedes_defns)

[{'category': 'Premises',
  'definition': 'Date a premises is expected to achieve assessment '
                'recognition, including in the appropriate cases, third party '
                'verification',
  'name': 'Assessment Compliance Target Date'},
 {'category': 'Premises',
  'definition': 'Eligibility of a premises for assessment recognition.',
  'name': 'Assessment Eligibility'},
 {'category': 'Premises',
  'definition': 'Value from assessment programs that produce a descriptive '
                '(rather than numeric) rating, such as LEED or NGBS.',
  'name': 'Assessment Level'},
 {'category': 'Premises',
  'definition': 'Program which issues energy labels, ratings, or '
                'sustainability certifications.',
  'name': 'Assessment Program'},
 {'category': 'Premises',
  'definition': 'The name of the body or group providing the verification or '
                'certification assessment program. More than one can apply to '
                'a premises.',
  'name': 'A

In [14]:
"""Get BuildingSync defns"""
buildingsync_parser = BuildingSyncParser()
buildingsync_defns = buildingsync_parser._get_term_definitions()

pprint(buildingsync_defns)

[{'definition': '', 'name': 'BuildingSync', 'tree': '/xs:schema/xs:element[1]'},
 {'definition': '',
  'name': 'Programs',
  'tree': '/xs:schema/xs:element[1]/xs:complexType/xs:sequence/xs:element[1]'},
 {'definition': 'Authorized or supported program such as rebate or audit.',
  'name': 'Program',
  'tree': '/xs:schema/xs:element[1]/xs:complexType/xs:sequence/xs:element[1]/xs:complexType/xs:sequence/xs:element'},
 {'definition': 'Date associated with the program.',
  'name': 'ProgramDate',
  'tree': '/xs:schema/xs:element[1]/xs:complexType/xs:sequence/xs:element[1]/xs:complexType/xs:sequence/xs:element/xs:complexType/xs:sequence/xs:element[1]'},
 {'definition': 'The source of funding or sponsor of the program.',
  'name': 'ProgramFundingSource',
  'tree': '/xs:schema/xs:element[1]/xs:complexType/xs:sequence/xs:element[1]/xs:complexType/xs:sequence/xs:element/xs:complexType/xs:sequence/xs:element[2]'},
 {'definition': 'The classification or type of the program.',
  'name': 'ProgramClas

In [15]:
"""Create Mapper"""
mapper = Mapper(
    first_definitions=[{"name": d["name"]} for d in buildingsync_defns],
    first_index_file="./indices/names/buidingsync.ollama.index",
    second_definitions=[{"name": d["name"]} for d in bedes_defns],
    second_index_file="./indices/names/bedes.ollama.index",
)

2025-02-26 16:11:24.139 | INFO     | brickmapper:populate_external_embeddings:48 - Restoring ./indices/names/buidingsync.ollama.index...
2025-02-26 16:11:24.172 | INFO     | brickmapper:populate_external_embeddings:48 - Restoring ./indices/names/bedes.ollama.index...


In [16]:
"""Get Mappings"""
top_k = 3
threshold = .3
results = mapper.get_mappings_with_collisions(top_k, threshold)

pd.DataFrame(
    [[name, *r] for name, r in results.items()], 
    columns=["BuildingSync Term", "#1 Best Match", "#2 Best Match", "#3 Best Match"]
).to_excel("BEDES-names-results.xlsx")

results

{'BuildingSync': [('Buildings Performance Database Tool', np.float32(0.29)),
  ('Building', np.float32(0.29)),
  ('Building', np.float32(0.29))],
 'Programs': [],
 'Program': [],
 'ProgramDate': [('Date', np.float32(0.2)),
  ('Date Status', np.float32(0.28)),
  ('Program', np.float32(0.29))],
 'ProgramFundingSource': [('Funding Source', np.float32(0.18)),
  ('Funded', np.float32(0.25)),
  ('Funding Amount', np.float32(0.27))],
 'ProgramClassification#Audit': [('Program Classification', np.float32(0.2)),
  ('Audit', np.float32(0.24)),
  ('Audit', np.float32(0.24))],
 'ProgramClassification#Performance': [('Program Classification',
   np.float32(0.17)),
  ('Efficiency Metric Qualifier', np.float32(0.29))],
 'ProgramClassification#Deemed': [],
 'ProgramClassification#Retrofit': [('Retrofit', np.float32(0.25)),
  ('Retrofit', np.float32(0.25)),
  ('Program Classification', np.float32(0.25))],
 'ProgramClassification#Rebate': [('Rebate', np.float32(0.17)),
  ('Rebate', np.float32(0.17)),
  

In [ ]:
"""Some quick analysis"""

number_of_suggestions_above_threshold = [len(r) for r in results.values()]
print(f"the threshold was a distance of {threshold}:")
for num_results, num_with_num_results in Counter(number_of_suggestions_above_threshold).items():
    print(f"\t {num_with_num_results} defintitons had {num_results} results below threshold")

print("\n\nhere are some of the defintions with no matches:")
pprint([name for name, r in results.items() if len(r) == 0][:5])

scores_by_rank = defaultdict(list)
for r in results.values():
    for rank, (_, score) in enumerate(r):
        scores_by_rank[rank].append(score)

print("\n\nhere's a histogram of score, groups by the rank of the match")
for rank in range(top_k):
    pyplot.hist(scores_by_rank[rank], alpha=0.5, label=rank+1)
    
pyplot.legend(loc='upper right')
pyplot.show()



In [ ]:
for rank in range(top_k):
    results_with_rank = [r for r in results.items() if len(r[1]) > rank]
    best_matches = sorted(results_with_rank, key=lambda x: 1 * x[1][rank][1])
    print(f"Some of the best matches for rank {rank + 1} are:")
    pprint([(m[0], m[1][rank]) for m in best_matches[:5]])
    print("\n")


In [ ]:
unmatched_terms = [term for term, matches in results.items() if len(matches) == 0]

pprint(unmatched_terms)

In [ ]:
set_of_broken_unmatched_terms = set()
for t in unmatched_terms:
    broken_term = re.findall('[A-Z][^A-Z]*', t)
    set_of_broken_unmatched_terms.update(broken_term)

pprint(set_of_broken_unmatched_terms)

In [ ]:
seen_broken_unmatched_terms = {t: results[t] for t in set_of_broken_unmatched_terms if t in results}

pprint(f"We have seen {len(seen_broken_unmatched_terms)} of the {len(set_of_broken_unmatched_terms)} broken unmatches terms in the original defintions")
pprint(seen_broken_unmatched_terms)

In [ ]:
unseen_broken_unmatched_terms = [t for t in set_of_broken_unmatched_terms if t not in results]

for term in unseen_broken_unmatched_terms[:10]:
    print(term)
    pprint(mapper.get_mappings_for_single_definition({"name": term, 'term_definition': ''}))
    